# Disease programme within CD14 monocytes — slide figure (2026-10-18)
Redraw of section 4 of `monocyte_gene_level.ipynb` for the deck (limit ④ detail slide). Input: `*_cd14mono_disease_logFC.csv.gz` (per-sample pseudobulk, case vs control, real A+B untouched vs generated).
Real disease genes = expressed genes (mean CP10k ≥ 1) with |log2FC| ≥ 1 and BH q < 0.05 in real; slope = OLS of generated on real over those genes.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
fig, axes = plt.subplots(1, 2, figsize=(12, 5.4)); rows = []
for ax, (c, lab) in zip(axes, [('sepsis', 'Sepsis vs HC'), ('covid', 'COVID+ vs COVID−')]):
    d = pd.read_csv(f'{c}_cd14mono_disease_logFC.csv.gz'); e = d[d.expressed]
    s = e[(e['log2FC|real'].abs() >= 1) & (e['q|real'] < 0.05)]
    ax.scatter(e['log2FC|real'], e['log2FC|gen K1,000'], s=6, c='#C9CED8', label=f'expressed genes ({len(e)})')
    ax.scatter(s['log2FC|real'], s['log2FC|gen K1,000'], s=26, c='#2A7AB8', label=f'real disease genes ({len(s)}), K = 1,000')
    ax.scatter(s['log2FC|real'], s['log2FC|gen K4,000'], s=26, c='#D7191C', marker='^', label='same genes, K = 4,000')
    lim = 5.5; ax.plot([-lim, lim], [-lim, lim], ls=':', c='k', lw=0.8); ax.axhline(0, c='k', lw=0.5); ax.axvline(0, c='k', lw=0.5)
    out = {'cohort': c, 'n_expressed': len(e), 'n_disease_genes': len(s)}
    for k in ['K1,000', 'K4,000']:
        x, y = s['log2FC|real'], s[f'log2FC|gen {k}']
        out[f'sign agree {k}'] = float(np.mean(np.sign(x) == np.sign(y))); out[f'slope {k}'] = float(np.polyfit(x, y, 1)[0])
        out[f'spearman all expressed {k}'] = float(e[['log2FC|real', f'log2FC|gen {k}']].corr('spearman').iloc[0, 1])
    rows.append(out)
    for g in ['IL1R2', 'S100A8', 'HLA-DRA', 'HLA-DPB1', 'MCEMP1', 'IFI27', 'ISG15', 'CD163', 'CLU']:
        r = s[s.gene == g]
        if len(r): ax.annotate(g, (r['log2FC|real'].iloc[0], r['log2FC|gen K1,000'].iloc[0]), fontsize=9, xytext=(3, 3), textcoords='offset points')
    ax.set_xlim(-3, 5.6); ax.set_ylim(-3, 5.6)
    ax.set_title(f"{lab}: sign {out['sign agree K1,000']:.0%}, slope {out['slope K1,000']:.2f} / {out['slope K4,000']:.2f}", fontsize=11)
    ax.set_xlabel('real disease log2FC (per-sample pseudobulk)'); ax.set_ylabel('generated disease log2FC')
axes[0].legend(frameon=False, fontsize=9, loc='upper left')
plt.tight_layout(); plt.savefig('disease_programme_slide.png', dpi=200); plt.show()
t = pd.DataFrame(rows).round(3); t.to_csv('disease_programme_slide_summary.csv', index=False); t